In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [3]:
def llm(prompt):
    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=prompt
    )
    return response.output_text

In [4]:
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

Yes — in most cases you can still join if the course is open or if late enrollment is allowed.

A few things to check:
- **Enrollment deadline**: some courses close registration after a set date
- **Prerequisites**: make sure you meet any required background
- **Catch-up work**: if the course has already started, you may need to make up missed material
- **Availability**: there may be limited seats

If you want, I can help you draft a short message to the instructor or course organizer asking whether you can still join.


In [5]:
context = '''
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram &amp; Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.

edit on GitHub
#Cloud alternatives with GPU
Check the quota and reset cycle carefully. Is the free hours limit per month or per week? Usually, if you change the configuration, the free hours quota might also be adjusted, or it might be billed separately.

Potential options include:

Google Colab
Kaggle
Databricks (possibly)
Consider using GPTs to discover more options. Be aware that some platforms might have restrictions on what you can and cannot install, so ensure to read what is included in the free vs paid tier.
'''

In [6]:
prompt = f'''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
'''

In [7]:
question = 'I just discovered the course. Can I join now?'
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want a certificate, you need to submit your project while submissions are still open.


In [8]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [9]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [10]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f'{url_prefix}{course['path']}'

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

In [11]:
documents[1100]

{'id': 'f580e98fdc',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 5. Deploying Machine Learning Models',
 'question': 'Waitress on Windows / Git Bash: "waitress-serve: command not found"',
 'answer': '`pip install waitress` does install `waitress-serve.exe`, but Python\'s `Scripts/` directory may not be on Git Bash\'s `PATH`. The pip output usually warns about this:\n\n```\nWARNING: The script waitress-serve.exe is installed in \'C:\\Users\\<you>\\...\\Scripts\'\nwhich is not on PATH.\n```\n\nTo fix, add that `Scripts` directory to Git Bash\'s `PATH` permanently:\n\n```bash\nnano ~/.bashrc\n# add this line, with the path from the warning:\nexport PATH="/c/Users/<you>/.../Scripts:$PATH"\n```\n\nClose Git Bash and reopen it. `waitress-serve --help` should now work.\n\nIf you\'re using `uv`, this isn\'t an issue because `uv run waitress-serve ...` runs the binary directly from the venv without needing it on `PATH`.'}

In [12]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)
index.fit(documents)

In [13]:
search_results = index.search(
    question,
    boost_dict={'question': 2.0, 'section': 0.5},
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c